# YOLO26 Training Pipeline

**Pipeline stages:**
1. **Stage 1** — Detection training on source domain (resin, 1 class)
2. **Stage 2** — Detection fine-tuning on target domain (pollen, 52 classes)
3. **Stage 3** — Classification training on target domain (pollen, 52 classes)

**Models:** n, s, m, l, x  
**Framework:** Ultralytics

## 0. Imports & Setup

In [1]:
import os
import json
from pathlib import Path
from ultralytics import YOLO

!echo $CUDA_VISIBLE_DEVICES
if os.environ.get("CUDA_VISIBLE_DEVICES") != "0,1":
    os.environ["CUDA_VISIBLE_DEVICES"] = "0,1" # Append for each next GPU
    !echo $CUDA_VISIBLE_DEVICES

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42

# ── Dataset YAMLs ──────────────────────────────────────────────────────────────
SOURCE_YAML   = "dataset_1_resin.yaml"   # resin  — 1 class,  source domain
TARGET_YAML   = "2025_pollen.yaml"        # pollen — 52 classes, target domain

# ── Model sizes to iterate ─────────────────────────────────────────────────────
SIZES = ["n", "s", "m", "l", "x"]

# ── Fixed hyper-parameters ─────────────────────────────────────────────────────
FIXED = dict(
    epochs     = 300,
    patience   = 50,
    batch      = 16,
    imgsz      = 640,
    workers    = 2,
    optimizer  = "SGD",
    momentum   = 0.937,
    weight_decay = 0.0005,
    cos_lr     = True,      # cosine LR scheduler
    lrf        = 0.01,      # final LR = lr0 * lrf
    seed       = SEED,
    deterministic = True,
    exist_ok  = True,
    verbose    = True,
    device     = "0",
)

# ── Per-stage learning rates ────────────────────────────────────────────────────
LR_SOURCE = 0.01    # Stage 1: train from scratch on source
LR_TARGET = 0.001   # Stage 2 & 3: fine-tune, weights already pretrained

# ── Output root ───────────────────────────────────────────────────────────────
print("Setup complete.")
print(f"Models to train: {SIZES}")
print(f"Fixed params: {json.dumps(FIXED, indent=2)}")

GPU-a499a47b-6eca-00c0-ef67-07b64dce9e09
0,1
Setup complete.
Models to train: ['n', 's', 'm', 'l', 'x']
Fixed params: {
  "epochs": 300,
  "patience": 50,
  "batch": 16,
  "imgsz": 640,
  "workers": 2,
  "optimizer": "SGD",
  "momentum": 0.937,
  "weight_decay": 0.0005,
  "cos_lr": true,
  "lrf": 0.01,
  "seed": 42,
  "deterministic": true,
  "exist_ok": true,
  "verbose": true,
  "device": "0"
}


In [2]:
RUNS_DIR = Path("runs/detect")

---
## Stage 1 — Detection: Source Domain (Resin, 1 class)

Train each YOLO26 size from its pretrained weights on the resin detection dataset.

In [3]:
stage1_results = {}
for size in SIZES:
    run_name = f"yolo26{size}_detect_source"
    print(f"\n{'='*60}")
    print(f" Stage 1 | yolo26{size} | Detection | Source (resin)")
    print(f"{'='*60}")
    model = YOLO(f"yolo26{size}.pt")
    results = model.train(
        data       = SOURCE_YAML,
        single_cls = True,
        lr0        = LR_SOURCE,
        name       = run_name,
        project    = str("stage1_detect_source"),
        **FIXED,
    )
    stage1_results[size] = results
    print(f"\n[Stage 1 | {size}] Best weights: {model.trainer.best}")
print("\n✅ Stage 1 complete for all sizes.")


 Stage 1 | yolo26n | Detection | Source (resin)
New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.10.12 torch-2.3.0a0+6ddf5cf85e.nv24.04 CUDA:0 (NVIDIA A40, 45488MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=dataset_1_resin.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.

---
## Stage 2 — Detection Fine-tuning: Target Domain (Pollen, 52 classes)

Load the best Stage 1 weights for each size and fine-tune on the pollen detection dataset.

In [ ]:
stage2_results = {}

for size in SIZES:
    # Resolve best weights from Stage 1
    stage1_best = (
        RUNS_DIR / "stage1_detect_source"
        / f"yolo26{size}_detect_source" / "weights" / "best.pt"
    )
    assert stage1_best.exists(), (
        f"Stage 1 weights not found: {stage1_best}\n"
        "Run Stage 1 first."
    )

    run_name = f"yolo26{size}_detect_target"
    print(f"\n{'='*60}")
    print(f" Stage 2 | yolo26{size} | Detection Fine-tune | Target (pollen)")
    print(f"{'='*60}")
    print(f" Loading weights: {stage1_best}")

    model = YOLO(str(stage1_best))

    results = model.train(
        data            = TARGET_YAML,
        single_cls      = True,
        lr0             = LR_TARGET,
        name            = run_name,
        project         = str("stage2_detect_target"),
        **FIXED,
    )

    stage2_results[size] = results
    print(f"\n[Stage 2 | {size}] Best weights: {model.trainer.best}")

print("\n✅ Stage 2 complete for all sizes.")


 Stage 2 | yolo26n | Detection Fine-tune | Target (pollen)
 Loading weights: runs/detect/stage1_detect_source/yolo26n_detect_source/weights/best.pt
New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.10.12 torch-2.3.0a0+6ddf5cf85e.nv24.04 CUDA:0 (NVIDIA A40, 45488MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=2025_pollen.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0

---
## Stage 3 — Classification: Target Domain (Pollen, 52 classes)

Switch task to classification. Load Stage 2 fine-tuned weights and train a classifier on the pollen dataset.

> **Note:** Ultralytics classification expects the dataset to be in `ImageFolder` format (one subfolder per class). Make sure your pollen classification split is organised accordingly, or point `data` to the correct root path.

In [ ]:
# ── Classification dataset root ─────────────────────────────────────────────────
# Expected structure:
#   CLF_DATA_ROOT/
#     train/  class_1/  *.jpg ...
#             class_2/  *.jpg ...
#     val/    class_1/  *.jpg ...
#             class_2/  *.jpg ...

stage3_results = {}

for size in SIZES:
    # Resolve best weights from Stage 2
    stage2_best = (
        RUNS_DIR / "stage2_detect_target"
        / f"yolo26{size}_detect_target" / "weights" / "best.pt"
    )
    assert stage2_best.exists(), (
        f"Stage 2 weights not found: {stage2_best}\n"
        "Run Stage 2 first."
    )

    run_name = f"yolo26{size}_classify_target"
    print(f"\n{'='*60}")
    print(f" Stage 3 | yolo26{size} | Classification | Target (pollen)")
    print(f"{'='*60}")
    print(f" Loading weights: {stage2_best}")

    model = YOLO(str(stage2_best))

    results = model.train(
        data    = TARGET_YAML,
        single_cls    = False,
        lr0     = LR_TARGET,
        name    = run_name,
        project = str("stage3_classify_target"),
        **FIXED,
    )

    stage3_results[size] = results
    print(f"\n[Stage 3 | {size}] Best weights: {model.trainer.best}")

print("\n✅ Stage 3 complete for all sizes.")

---
## Results Summary

In [ ]:
import pandas as pd

def get_metric(results_dict, size, metric):
        try:
            r = results_dict[size]
            return round(float(getattr(r, metric, None) or r.results_dict.get(metric, float('nan'))), 4)
        except Exception:
            return None

rows = []
for size in SIZES:

    # Stage 1 & 2: detection → mAP50
    s1_map = get_metric(stage1_results, size, "metrics/mAP50(B)")
    s2_map = get_metric(stage2_results, size, "metrics/mAP50(B)")
    s3_map = get_metric(stage3_results, size, "metrics/mAP50(B)")

    rows.append({
        "Model": f"yolo26{size}",
        "S1 Detect mAP50 (source)": s1_map,
        "S2 Detect mAP50 (target)": s2_map,
        "S3 Classify mAP50 (target)": s3_map,
    })

df = pd.DataFrame(rows).set_index("Model")
print(df.to_string())
df

In [ ]:
print(stage3_results["n"].results_dict)

In [ ]:
# Save summary to CSV
summary_path = RUNS_DIR / "training_summary.csv"
df.to_csv(summary_path)
print(f"Summary saved to {summary_path}")